# 07 — 选股推荐

> **先决条件**：06 板块分析已就绪 + C++ 路径已修复（Module C++ 可选）

流程：板块动量筛选 → 成分股三维过滤 → C++ 打分 → 汇总排名 → 推荐卡片

| 维度 | 来源 | 满分 |
|------|------|------|
| 板块强度 | 板块20日涨幅百分位 | 20 |
| 资金流入 | large_order_net_ratio 5日均 | 20 |
| 基本面质量 | ROE + 利润增速 | 20 |
| 个股超额收益 | 近20日超额收益 | 20 |
| C++ 预测分 | bull_prob × 20（可选）| 20 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  参数单元（修改这里即可调整推荐逻辑）
# ══════════════════════════════════════════════════════════════════════════════
FORECAST_DAYS    = 20     # 预测窗口（交易日）
TOP_SECTORS      = 3      # 取前 N 个强势板块
MAX_PICKS        = 5      # 最多推荐几只
MIN_ROE          = 0.10   # 基本面门槛 ROE 下限（小数，如 0.10 = 10%）
MIN_FUND_FLOW    = 0.0    # 大单净流入比门槛（0 = 不过滤）
SECTOR_WINDOW    = 20     # 板块动量计算窗口（天）
EXCESS_WINDOW    = 20     # 个股超额收益窗口（天）
FF_LOOKBACK      = 5      # 资金流均值窗口（天）

In [ ]:
# ── 初始化 ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / 'data'

import pandas as pd
import numpy as np
import sqlite3
from datetime import date, timedelta
from IPython.display import display, HTML

from trade_py.data.market.index.tushare import SW_SECTOR_INDICES
from trade_py.analysis.knowledge_graph import SW, SW_NAMES_ZH

INDEX_DIR   = DATA_ROOT / 'index'
KLINE_DIR   = DATA_ROOT / 'kline'
FUND_DIR    = DATA_ROOT / 'fundamental'
FF_DIR      = DATA_ROOT / 'fund_flow'
TODAY       = date.today()

print(f'项目: {PROJECT_ROOT}')
print(f'日期: {TODAY}  预测窗口: {FORECAST_DAYS} 交易日')

## Section 1 — 板块筛选（取强势板块）

In [ ]:
sector_data = {}
for code, (zh_name, sw_idx) in SW_SECTOR_INDICES.items():
    safe = 'sector_' + code.replace('.', '_') + '.parquet'
    p = INDEX_DIR / safe
    if p.exists():
        try:
            df = pd.read_parquet(p)
            df['_date'] = pd.to_datetime(df['date'])
            sector_data[code] = df.sort_values('_date').reset_index(drop=True)
        except Exception:
            pass

if not sector_data:
    print('⚠ 无板块指数数据，请先运行: ./trade data index sync-sector --start 2023-01-01')
else:
    # 计算板块动量
    momentum_rows = []
    for code, df in sector_data.items():
        zh_name, sw_idx = SW_SECTOR_INDICES[code]
        if len(df) < SECTOR_WINDOW + 1:
            continue
        p_now  = float(df['close'].iloc[-1])
        p_past = float(df['close'].iloc[-SECTOR_WINDOW - 1])
        ret = (p_now - p_past) / p_past * 100 if p_past > 0 else 0.0
        momentum_rows.append({'code': code, 'sector': zh_name, 'sw_idx': sw_idx,
                               f'{SECTOR_WINDOW}日涨幅%': round(ret, 2)})

    mom_df = pd.DataFrame(momentum_rows).sort_values(f'{SECTOR_WINDOW}日涨幅%', ascending=False)

    # 百分位得分 0-100
    vals = mom_df[f'{SECTOR_WINDOW}日涨幅%'].values
    mom_df['momentum_pct'] = pd.Series(vals).rank(pct=True).values

    top_sectors = mom_df.head(TOP_SECTORS)
    print(f'Top {TOP_SECTORS} 强势板块（近{SECTOR_WINDOW}日）:')
    display(top_sectors[['sector', f'{SECTOR_WINDOW}日涨幅%', 'momentum_pct']])

## Section 2 — 候选股池生成

In [ ]:
if not sector_data:
    print('跳过')
else:
    db_path = DATA_ROOT / '.metadata' / 'trade.db'
    if not db_path.exists():
        print('⚠ trade.db 不存在')
    else:
        with sqlite3.connect(str(db_path)) as conn:
            df_instr = pd.read_sql('SELECT symbol, name, industry FROM instruments', conn)

        top_sw_indices = set(top_sectors['sw_idx'].tolist())
        candidates = df_instr[df_instr['industry'].isin(top_sw_indices)].copy()
        candidates['sector'] = candidates['industry'].apply(
            lambda x: SW_NAMES_ZH.get(SW(int(x)), '未知')
        )
        print(f'板块成分股候选: {len(candidates)} 只')

        # 过滤：K线文件必须存在
        valid_candidates = []
        for _, row in candidates.iterrows():
            kf = KLINE_DIR / (row['symbol'].replace('.', '_') + '.parquet')
            if kf.exists():
                valid_candidates.append(row)
        candidates = pd.DataFrame(valid_candidates).reset_index(drop=True)
        print(f'有 K 线数据: {len(candidates)} 只')
        display(candidates.head(10))

In [ ]:
# 三维过滤：ROE + 大单净流入 + 超额收益
if 'candidates' not in dir() or len(candidates) == 0:
    print('跳过')
else:
    scores = []

    # ── 反查对应板块指数
    ind_to_code = {sw_idx: code for code, (zh_name, sw_idx) in SW_SECTOR_INDICES.items()}

    for _, row in candidates.iterrows():
        sym = row['symbol']
        entry = {'symbol': sym, 'name': row['name'], 'sector': row['sector']}

        # 1. 超额收益
        kf = KLINE_DIR / (sym.replace('.', '_') + '.parquet')
        excess_ret = None
        try:
            kl = pd.read_parquet(kf)
            kl['_date'] = pd.to_datetime(kl['date'])
            kl = kl.sort_values('_date').tail(EXCESS_WINDOW + 1)
            if len(kl) >= 2:
                stock_ret = (kl['close'].iloc[-1] - kl['close'].iloc[0]) / kl['close'].iloc[0] * 100
                sec_code = ind_to_code.get(int(row['industry']))
                sec_ret = 0.0
                if sec_code and sec_code in sector_data:
                    sd = sector_data[sec_code].tail(EXCESS_WINDOW + 1)
                    if len(sd) >= 2:
                        sec_ret = (sd['close'].iloc[-1] - sd['close'].iloc[0]) / sd['close'].iloc[0] * 100
                excess_ret = stock_ret - sec_ret
        except Exception:
            pass
        entry['excess_ret'] = round(excess_ret, 2) if excess_ret is not None else None

        # 2. 资金流
        avg_net_flow = None
        ff_path = FF_DIR / (sym.replace('.', '_') + '.parquet')
        if ff_path.exists():
            try:
                ff = pd.read_parquet(ff_path)
                net_col = next((c for c in ['buy_lg_amount', 'net_mf_amount', 'large_order_net_ratio'] if c in ff.columns), None)
                if net_col:
                    avg_net_flow = float(pd.to_numeric(ff[net_col], errors='coerce').dropna().tail(FF_LOOKBACK).mean())
            except Exception:
                pass
        entry['avg_net_flow'] = round(avg_net_flow, 4) if avg_net_flow is not None else None

        # 3. 基本面 ROE
        roe = None
        fund_path = FUND_DIR / (sym.replace('.', '_') + '.parquet')
        if fund_path.exists():
            try:
                fd = pd.read_parquet(fund_path)
                roe_col = next((c for c in ['roe', 'roe_yearly', 'roe_dt'] if c in fd.columns), None)
                if roe_col:
                    roe = float(pd.to_numeric(fd[roe_col], errors='coerce').dropna().iloc[-1])
            except Exception:
                pass
        entry['roe'] = round(roe, 2) if roe is not None else None

        scores.append(entry)

    df_scores = pd.DataFrame(scores)
    print(f'特征计算完成: {len(df_scores)} 只')
    display(df_scores.head())

## Section 3 — C++ 打分（可选）

In [ ]:
from trade_py.ui.services import cpp_bridge

CPP_AVAILABLE = cpp_bridge.cli_available()
if CPP_AVAILABLE:
    print(f'✅ trade_cli: {cpp_bridge._TRADE_CLI}')
else:
    print(f'⚠ trade_cli 不可用，C++ 预测分将设为 0（路径: {cpp_bridge._TRADE_CLI}）')

if 'df_scores' in dir() and CPP_AVAILABLE:
    bull_probs = {}
    for sym in df_scores['symbol'].tolist():
        result = cpp_bridge.get_predict(sym)
        if 'bull_prob' in result:
            bull_probs[sym] = float(result['bull_prob'])
        elif 'error' not in result:
            # 尝试从 features 中取
            feat = cpp_bridge.get_features(sym)
            bull_probs[sym] = float(feat.get('bull_prob', 0.5))
    df_scores['bull_prob'] = df_scores['symbol'].map(bull_probs).fillna(0.5)
    print(f'C++ 预测分获取: {len(bull_probs)} 只')
else:
    if 'df_scores' in dir():
        df_scores['bull_prob'] = 0.5  # 默认中性

## Section 4 — 汇总打分 + 排名

In [ ]:
if 'df_scores' not in dir() or len(df_scores) == 0:
    print('无候选股，跳过')
else:
    # 合并板块动量得分
    df_scores['sw_idx'] = df_scores['symbol'].map(
        dict(zip(df_instr['symbol'], df_instr['industry']))
    )
    sw_to_mom_pct = dict(zip(mom_df['sw_idx'], mom_df['momentum_pct']))
    df_scores['momentum_pct'] = df_scores['sw_idx'].map(sw_to_mom_pct).fillna(0.5)

    # 五维打分（0-20 各）
    def _normalize(series, reverse=False):
        s = pd.to_numeric(series, errors='coerce')
        mn, mx = s.min(), s.max()
        if mx == mn:
            return pd.Series([10.0] * len(s), index=s.index)
        norm = (s - mn) / (mx - mn)
        if reverse:
            norm = 1 - norm
        return norm * 20

    df_scores['score_sector']   = _normalize(df_scores['momentum_pct'])
    df_scores['score_fund']     = _normalize(df_scores['avg_net_flow'].fillna(0))
    df_scores['score_roe']      = _normalize(df_scores['roe'].fillna(0))
    df_scores['score_excess']   = _normalize(df_scores['excess_ret'].fillna(0))
    df_scores['score_cpp']      = df_scores['bull_prob'].fillna(0.5) * 20

    df_scores['total_score'] = (
        df_scores['score_sector']
        + df_scores['score_fund']
        + df_scores['score_roe']
        + df_scores['score_excess']
        + df_scores['score_cpp']
    ).round(1)

    # 基本面过滤
    df_final = df_scores.copy()
    if MIN_ROE > 0:
        roe_mask = df_final['roe'].isna() | (df_final['roe'] >= MIN_ROE * 100)
        df_final = df_final[roe_mask]
    if MIN_FUND_FLOW > 0:
        ff_mask = df_final['avg_net_flow'].isna() | (df_final['avg_net_flow'] >= MIN_FUND_FLOW)
        df_final = df_final[ff_mask]

    top_picks = df_final.sort_values('total_score', ascending=False).head(MAX_PICKS)

    cols_show = ['symbol', 'name', 'sector', 'total_score',
                 'score_sector', 'score_fund', 'score_roe', 'score_excess', 'score_cpp',
                 'roe', 'avg_net_flow', 'excess_ret', 'bull_prob']
    display(top_picks[[c for c in cols_show if c in top_picks.columns]])

    import plotly.express as px
    if not top_picks.empty:
        score_cols = ['score_sector', 'score_fund', 'score_roe', 'score_excess', 'score_cpp']
        score_names = ['板块强度', '资金流入', '基本面', '超额收益', 'C++预测']
        fig = go.Figure()
        for _, row in top_picks.iterrows():
            vals = [float(row.get(c, 0)) for c in score_cols]
            fig.add_trace(go.Scatterpolar(
                r=vals + [vals[0]],
                theta=score_names + [score_names[0]],
                name=f"{row['symbol']} {row['name']}",
                fill='toself', opacity=0.5,
            ))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 20])),
                          title='Top 推荐股票 五维雷达图')
        fig.show()

## Section 5 — 推荐卡片输出

In [ ]:
if 'top_picks' not in dir() or len(top_picks) == 0:
    print('无推荐股票')
else:
    medals = ['🏆', '🥈', '🥉', '4️⃣', '5️⃣']

    print('=' * 65)
    print(f'  推荐时间: {TODAY}    预测窗口: {FORECAST_DAYS} 个交易日')
    print('=' * 65)

    for rank, (_, row) in enumerate(top_picks.iterrows()):
        medal = medals[rank] if rank < len(medals) else f'#{rank+1}'
        roe_str   = f"{row['roe']:.1f}%" if pd.notna(row.get('roe')) else '无数据'
        flow_str  = f"{row['avg_net_flow']:.4f}" if pd.notna(row.get('avg_net_flow')) else '无数据'
        excess_str = f"{row['excess_ret']:+.2f}%" if pd.notna(row.get('excess_ret')) else '无数据'
        bull_str  = f"{row['bull_prob']*100:.0f}%" if CPP_AVAILABLE else '不可用'

        # 板块动量
        sec_code = ind_to_code.get(int(row.get('sw_idx', 0)))
        sec_mom = mom_df[mom_df['code'] == sec_code][f'{SECTOR_WINDOW}日涨幅%'].values
        sec_mom_str = f"+{sec_mom[0]:.1f}%" if len(sec_mom) > 0 and sec_mom[0] >= 0 else (f"{sec_mom[0]:.1f}%" if len(sec_mom) > 0 else '—')

        print(f'\n{medal} #{rank+1}  {row["symbol"]}  {row["name"]}          综合评分: {row["total_score"]:.0f}/100')
        print(f'   板块: {row["sector"]}（近{SECTOR_WINDOW}日 {sec_mom_str}）')
        print(f'   资金: 大单净流入均值 {flow_str}')
        print(f'   基本面: ROE {roe_str}')
        print(f'   技术: {EXCESS_WINDOW}日超额收益 {excess_str}')
        print(f'   C++ 预测: 多头概率 {bull_str}')
        print()

    print('=' * 65)

## Section 6 — 推荐历史追踪

In [ ]:
import os

PICKS_DIR = DATA_ROOT / 'picks'
PICKS_DIR.mkdir(parents=True, exist_ok=True)

if 'top_picks' in dir() and len(top_picks) > 0:
    save_path = PICKS_DIR / f'{TODAY.strftime("%Y-%m-%d")}.csv'
    save_cols = ['symbol', 'name', 'sector', 'total_score',
                 'roe', 'avg_net_flow', 'excess_ret', 'bull_prob']
    top_picks[[c for c in save_cols if c in top_picks.columns]].to_csv(save_path, index=False)
    print(f'✅ 推荐记录已保存: {save_path}')

    # 显示历史记录
    history_files = sorted(PICKS_DIR.glob('*.csv'), reverse=True)[:10]
    if history_files:
        print(f'\n历史推荐记录（最近 {len(history_files)} 次）:')
        hist_dfs = []
        for f in history_files:
            df = pd.read_csv(f)
            df['date'] = f.stem
            hist_dfs.append(df)
        hist_all = pd.concat(hist_dfs, ignore_index=True)
        display(hist_all.groupby(['date', 'symbol', 'name'])['total_score'].first().unstack('date').fillna('-'))
else:
    print('无推荐记录可保存')